In [23]:
import pandas as pd
import numpy as np

In [24]:
df = pd.read_csv("E:\CLV_Project\online_retail_10_11.csv")

df.head()

<>:1: SyntaxWarning: invalid escape sequence '\C'
<>:1: SyntaxWarning: invalid escape sequence '\C'
C:\Users\hp\AppData\Local\Temp\ipykernel_16160\1999927294.py:1: SyntaxWarning: invalid escape sequence '\C'
  df = pd.read_csv("E:\CLV_Project\online_retail_10_11.csv")


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [25]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [26]:
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

In [27]:
recency = df.groupby('CustomerID')['InvoiceDate'].max().reset_index()

recency['Recency'] = (reference_date - recency['InvoiceDate']).dt.days

recency = recency[['CustomerID', 'Recency']]

In [28]:
frequency = df.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()
frequency.columns = ['CustomerID', 'Frequency']

In [29]:
monetary = df.groupby('CustomerID')['Revenue'].sum().reset_index()

monetary.columns = ['CustomerID', 'Monetary']

In [32]:
rfm = recency.merge(frequency, on='CustomerID')
rfm = rfm.merge(monetary, on='CustomerID')

# optional, if you want the downstream cells (which use "Customer ID") to keep working:
rfm = rfm.rename(columns={'CustomerID': 'Customer ID'})

rfm.head()

,Customer ID,Recency,Frequency,Monetary
0,12346.0,326,1,77183.60
1,12347.0,2,7,4310.00
2,12348.0,75,4,1797.24
3,12349.0,19,1,1757.55
4,12350.0,310,1,334.40


In [34]:
# df has CustomerID, not Customer ID
avg_order_value = df.groupby('CustomerID')['Revenue'].mean().reset_index()
avg_order_value.columns = ['CustomerID', 'AvgOrderValue']

# align join key with rfm column name
avg_order_value = avg_order_value.rename(columns={'CustomerID': 'Customer ID'})

rfm = rfm.merge(avg_order_value, on='Customer ID')

In [36]:
lifespan = df.groupby('CustomerID')['InvoiceDate'].agg(['min', 'max']).reset_index()

lifespan['Lifespan'] = (lifespan['max'] - lifespan['min']).dt.days

lifespan = lifespan[['CustomerID', 'Lifespan']]

lifespan = lifespan.rename(columns={'CustomerID': 'Customer ID'})

rfm = rfm.merge(lifespan, on='Customer ID')

In [37]:
rfm['PurchaseFrequency'] = rfm['Frequency'] / (rfm['Lifespan'] + 1)

In [38]:
rfm['Recency_Frequency'] = rfm['Recency'] * rfm['Frequency']
rfm['Monetary_Frequency'] = rfm['Monetary'] * rfm['Frequency']

In [39]:
rfm = rfm[(rfm['Monetary'] < rfm['Monetary'].quantile(0.99))]

In [40]:
rfm['CLV_Segment'] = pd.qcut(rfm['Monetary'], q=3, labels=[0,1,2])

In [41]:
rfm.head()

,Customer ID,Recency,Frequency,Monetary,AvgOrderValue,Lifespan,PurchaseFrequency,Recency_Frequency,Monetary_Frequency,CLV_Segment
1,12347.0,2,7,4310.00,23.681319,365,0.019126,14,30170.00,2
2,12348.0,75,4,1797.24,57.975484,282,0.014134,300,7188.96,2
3,12349.0,19,1,1757.55,24.076027,0,1.000000,19,1757.55,2
4,12350.0,310,1,334.40,19.670588,0,1.000000,310,334.40,0
5,12352.0,36,8,2506.04,29.482824,260,0.030651,288,20048.32,2


In [42]:
rfm.shape

(4294, 10)

In [ ]:
import os

output_dir = r"E:/CLV_Project/data"
os.makedirs(output_dir, exist_ok=True)

rfm.to_csv(os.path.join(output_dir, "feature_engineered_data.csv"), index=False)

OSError: Cannot save file into a non-existent directory: 'E:\CLV_Project\data'